# 第16章: Transformer を最新環境で継続検証する

この Notebook は、読み取り専用の原本 `machine-learning-book/ch16/` を参照しながら、Chapter 16 の要点を `pytest --nbmake` で実行しやすい形に再構成したものです。
巨大な事前学習済みモデルのダウンロードや長時間学習は避け、自己注意、因果マスク、位置エンコーディング、そして transformer ベースの最小タスクをローカル完結で検証します。


## この Notebook で確認すること

- 現在の `uv` 環境で Chapter 16 に必要な主要パッケージが使えることを確認する。
- 原本図版を読み取り専用のサブモジュールから参照できることを確認する。
- 自己注意の重み計算を手計算と行列演算の両方で再現する。
- `nn.MultiheadAttention` による因果マスク付き注意と位置エンコーディングを確認する。
- GPT 系の「次トークン予測」と BERT 系の「下流タスク分類」を、小さなローカルデータで再現する。


In [ ]:
from importlib.metadata import version
from pathlib import Path
import math
import platform
import random
import sys

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image, display

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = next(
    (
        candidate.resolve()
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'machine-learning-book').exists()
    ),
    None,
)
assert REPO_ROOT is not None, 'machine-learning-book を含むリポジトリルートを見つけられませんでした'

FIG_DIR = REPO_ROOT / 'machine-learning-book/ch16/figures'
assert FIG_DIR.exists(), f'図版ディレクトリが見つかりません: {FIG_DIR}'

print(f'Python 実行ファイル: {sys.executable}')
print(f'Python バージョン: {platform.python_version()}')
print(f'Matplotlib バックエンド: {matplotlib.get_backend()}')
print(f'図版ディレクトリ: {FIG_DIR}')


In [ ]:
PACKAGE_NAMES = ['numpy', 'pandas', 'matplotlib', 'torch', 'pytest', 'nbmake']
package_versions = pd.DataFrame(
    [(name, version(name)) for name in PACKAGE_NAMES],
    columns=['パッケージ', 'バージョン'],
)
package_versions


## 原本図版の参照

Chapter 16 の概念図は原本サブモジュールに残し、移行後 Notebook から相対探索で参照します。
自己注意、transformer 全体像、GPT/BERT の位置づけを図版で確認します。


In [ ]:
selected_figures = [
    ('16_01.png', 460),
    ('16_06.png', 560),
    ('16_11.png', 620),
    ('16_13.png', 560),
]

for figure_name, width in selected_figures:
    figure_path = FIG_DIR / figure_name
    print(figure_path.name)
    display(Image(filename=str(figure_path), width=width))


## 自己注意の基本を手計算で確かめる

原本 Part 1 では、単語埋め込みの内積から注意重みを作る流れを段階的に示しています。
ここでは同じ流れを、小さな文に対して最新の PyTorch で再現します。


In [ ]:
token_names = ['can', 'help', 'me', 'sentence', 'this', 'to', 'translate', 'you']
sentence = torch.tensor([0, 7, 1, 2, 5, 6, 4, 3])
embed = nn.Embedding(10, 16)
embedded_sentence = embed(sentence).detach()

omega = torch.empty(sentence.numel(), sentence.numel())
for i, x_i in enumerate(embedded_sentence):
    for j, x_j in enumerate(embedded_sentence):
        omega[i, j] = torch.dot(x_i, x_j)

omega_mat = embedded_sentence @ embedded_sentence.T
assert torch.allclose(omega, omega_mat)

attention_weights = F.softmax(omega, dim=1)
context_vectors = attention_weights @ embedded_sentence

self_attention_summary = pd.DataFrame(
    {
        'token': [token_names[idx] for idx in sentence.tolist()],
        'self_weight': attention_weights.diag().round(decimals=4).tolist(),
        'context_norm': context_vectors.norm(dim=1).round(decimals=4).tolist(),
    }
)
self_attention_summary


In [ ]:
focus_token_index = 1
attention_view = pd.Series(
    attention_weights[focus_token_index].tolist(),
    index=[token_names[idx] for idx in sentence.tolist()],
    name='attention_weight',
).sort_values(ascending=False)

print("クエリ token:", token_names[sentence[focus_token_index].item()])
attention_view


## 学習可能な Query / Key / Value とスケールド内積注意

次に、原本と同様に `Query`, `Key`, `Value` の射影行列を導入し、`sqrt(d_k)` で割るスケールド内積注意を確認します。
後半では `nn.MultiheadAttention` を使い、将来トークンを見ない因果マスクも追加します。


In [ ]:
d_model = embedded_sentence.shape[1]
U_query = torch.rand(d_model, d_model)
U_key = torch.rand(d_model, d_model)
U_value = torch.rand(d_model, d_model)

x_2 = embedded_sentence[1]
query_2 = U_query @ x_2
keys = (U_key @ embedded_sentence.T).T
values = (U_value @ embedded_sentence.T).T

attention_scores_2 = (query_2 @ keys.T) / math.sqrt(d_model)
attention_weights_2 = F.softmax(attention_scores_2, dim=0)
context_vector_2 = attention_weights_2 @ values

pd.DataFrame(
    {
        'token': [token_names[idx] for idx in sentence.tolist()],
        'scaled_score': attention_scores_2.detach().round(decimals=3).tolist(),
        'attention_weight': attention_weights_2.detach().round(decimals=4).tolist(),
    }
).sort_values('attention_weight', ascending=False)


In [ ]:
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=4, batch_first=True)
attn_input = embedded_sentence.unsqueeze(0)
causal_mask = torch.triu(
    torch.ones(sentence.numel(), sentence.numel(), dtype=torch.bool),
    diagonal=1,
)

attn_output, attn_weights = mha(
    attn_input,
    attn_input,
    attn_input,
    attn_mask=causal_mask,
    need_weights=True,
    average_attn_weights=False,
)

assert attn_output.shape == (1, sentence.numel(), d_model)
assert attn_weights.shape == (1, 4, sentence.numel(), sentence.numel())
assert torch.allclose(
    attn_weights[0, :, 0, 1:],
    torch.zeros_like(attn_weights[0, :, 0, 1:]),
    atol=1e-6,
)

pd.DataFrame(
    attn_weights[0, 0].detach().numpy(),
    index=[token_names[idx] for idx in sentence.tolist()],
    columns=[token_names[idx] for idx in sentence.tolist()],
).round(3)


## 位置エンコーディング

Transformer は再帰構造を持たないため、語順を表す情報を別途埋め込む必要があります。
ここでは正弦波ベースの位置エンコーディングを自前実装し、周波数ごとのパターンをヒートマップで可視化します。


In [ ]:
def positional_encoding(length: int, d_model: int) -> torch.Tensor:
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(length, d_model)
    pe[:, 0::2] = torch.sin(positions * div_term)
    pe[:, 1::2] = torch.cos(positions * div_term)
    return pe

pe = positional_encoding(length=8, d_model=16)

fig, ax = plt.subplots(figsize=(7, 3))
heatmap = ax.imshow(pe.T, aspect='auto', cmap='viridis')
ax.set_title('正弦波ベース位置エンコーディング')
ax.set_xlabel('位置')
ax.set_ylabel('埋め込み次元')
fig.colorbar(heatmap, ax=ax, fraction=0.03, pad=0.02)
plt.show()
plt.close(fig)

pd.DataFrame(pe[:4, :6].numpy(), columns=[f'dim_{i}' for i in range(6)]).round(4)


## GPT 系を模した最小の次トークン予測

原本 Part 2 では GPT-2 を読み込み、事前学習済み言語モデルの挙動を確認しています。
CI で安定して動かすため、ここでは外部ダウンロードの代わりに、小さな文集合だけで学習する transformer 言語モデルを作り、因果マスク付きで「次の単語」を予測させます。


In [ ]:
corpus = [
    'transformers model long range dependencies',
    'attention helps models focus on relevant tokens',
    'masked attention prevents access to future tokens',
    'encoder representations support downstream classification',
    'decoder models predict the next token autoregressively',
    'pretraining on unlabeled text improves transfer performance',
] * 8

specials = ['<pad>', '<bos>', '<eos>']
vocab = {token: idx for idx, token in enumerate(specials)}
for text in corpus:
    for token in text.split():
        if token not in vocab:
            vocab[token] = len(vocab)

id_to_token = {idx: token for token, idx in vocab.items()}
sequences = [
    [vocab['<bos>'], *[vocab[token] for token in text.split()], vocab['<eos>']]
    for text in corpus
]
max_length = max(len(sequence) for sequence in sequences)

language_model_inputs = []
language_model_targets = []
for sequence in sequences:
    features = sequence[:-1]
    labels = sequence[1:]
    padding = max_length - 1 - len(features)
    language_model_inputs.append(features + [vocab['<pad>']] * padding)
    language_model_targets.append(labels + [vocab['<pad>']] * padding)

language_model_inputs = torch.tensor(language_model_inputs)
language_model_targets = torch.tensor(language_model_targets)

class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 32, num_heads: int = 4, max_len: int = 16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional = nn.Parameter(positional_encoding(max_len, d_model).unsqueeze(0), requires_grad=False)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=64,
            dropout=0.0,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        seq_len = tokens.size(1)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=tokens.device, dtype=torch.bool),
            diagonal=1,
        )
        padding_mask = tokens.eq(vocab['<pad>'])
        hidden = self.embedding(tokens) + self.positional[:, :seq_len, :]
        encoded = self.encoder(hidden, mask=causal_mask, src_key_padding_mask=padding_mask)
        return self.output(encoded)

language_model = TinyLanguageModel(len(vocab), max_len=max_length)
optimizer = torch.optim.Adam(language_model.parameters(), lr=0.03)

for epoch in range(80):
    optimizer.zero_grad()
    logits = language_model(language_model_inputs)
    loss = F.cross_entropy(
        logits.reshape(-1, len(vocab)),
        language_model_targets.reshape(-1),
        ignore_index=vocab['<pad>'],
    )
    loss.backward()
    optimizer.step()

loss_value = float(loss.detach())
assert loss_value < 0.5

prefix_tokens = torch.tensor(
    [[
        vocab['<bos>'],
        vocab['masked'],
        vocab['attention'],
        vocab['prevents'],
        vocab['access'],
        vocab['to'],
    ]]
)
with torch.no_grad():
    next_token_distribution = language_model(prefix_tokens)[0, -1].softmax(dim=0)

top_values, top_indices = torch.topk(next_token_distribution, k=5)
pd.DataFrame(
    {
        'token': [id_to_token[index.item()] for index in top_indices],
        'probability': top_values.detach().round(decimals=4).tolist(),
    }
)


## BERT 系を模した最小の分類タスク

原本 Part 3 は DistilBERT を IMDb にファインチューニングしますが、そのままではモデルとデータのダウンロードに依存します。
ここでは同じ「encoder 表現を下流分類へ使う」という学習意図を保ちつつ、軽量な transformer encoder をローカルの小さな感情文データで学習させます。


In [ ]:
examples = [
    ('this movie is excellent and inspiring', 1),
    ('what a fantastic and heartwarming film', 1),
    ('the story was enjoyable and uplifting', 1),
    ('the acting is dull and disappointing', 0),
    ('this film is boring and predictable', 0),
    ('the plot was weak and forgettable', 0),
] * 12
random.shuffle(examples)

classifier_vocab = {'<pad>': 0, '<unk>': 1}
for text, _ in examples:
    for token in text.split():
        if token not in classifier_vocab:
            classifier_vocab[token] = len(classifier_vocab)

sequence_length = max(len(text.split()) for text, _ in examples)
features = torch.tensor(
    [
        [classifier_vocab.get(token, 1) for token in text.split()] + [0] * (sequence_length - len(text.split()))
        for text, _ in examples
    ]
)
labels = torch.tensor([label for _, label in examples])

permutation = torch.randperm(len(features))
train_idx, test_idx = permutation[:60], permutation[60:]
X_train, X_test = features[train_idx], features[test_idx]
y_train, y_test = labels[train_idx], labels[test_idx]

class TinyEncoderClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 32, num_heads: int = 4, max_len: int = 16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.positional = nn.Parameter(positional_encoding(max_len, d_model).unsqueeze(0), requires_grad=False)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=64,
            dropout=0.0,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        seq_len = tokens.size(1)
        padding_mask = tokens.eq(0)
        hidden = self.embedding(tokens) + self.positional[:, :seq_len, :]
        encoded = self.encoder(hidden, src_key_padding_mask=padding_mask)
        valid_token_counts = (~padding_mask).sum(dim=1, keepdim=True)
        pooled = encoded.masked_fill(padding_mask.unsqueeze(-1), 0.0).sum(dim=1) / valid_token_counts
        return self.classifier(pooled)

classifier = TinyEncoderClassifier(len(classifier_vocab), max_len=sequence_length)
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.02)

for epoch in range(60):
    optimizer.zero_grad()
    logits = classifier(X_train)
    loss = F.cross_entropy(logits, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_logits = classifier(X_test)
    predictions = test_logits.argmax(dim=1)
    accuracy = (predictions == y_test).float().mean().item()

assert accuracy >= 0.9

pd.DataFrame(
    {
        'text': [examples[index][0] for index in test_idx[:8].tolist()],
        'label': y_test[:8].tolist(),
        'prediction': predictions[:8].tolist(),
    }
)


## まとめ

- 原本 Part 1 の自己注意と transformer 構成要素を、最新の PyTorch API で再現しました。
- 原本 Part 2 の GPT-2 例は、因果マスク付きの小さな次トークン予測モデルへ置き換えました。
- 原本 Part 3 の DistilBERT fine-tuning は、軽量な encoder 分類器によるローカル完結タスクへ置き換えました。
- いずれのセルも GUI や対話入力に依存せず、`nbmake` でヘッドレス実行できる構成です。
